# 01 — Fit3D → the same features and truth as the REHAB24-6 benchmark

Runs on your PC or on a Linux server (CPU only: MediaPipe has no GPU path in Python). On a PC the ~96 curl videos take roughly 1-2 hours; the cell can be stopped and re-run, finished videos are cached.

For every subject × camera × exercise it

1. runs MediaPipe Pose (image + world landmarks) — cached, parallel over videos;
2. computes the **true** elbow angle from Fit3D's motion-capture joints (`joints3d_25`);
3. finds which of the 25 joints are shoulder/elbow/wrist **from the data** (projects them with the camera file and matches them to MediaPipe);
4. labels the camera *front / half / profile* from the shoulder line;
5. builds variant A, B, E and the 20 I_ML features **with exactly the repo code the REHAB24-6 benchmark uses** (same visibility mask 0.70, gap fill 5, causal Butterworth 6 Hz, same inclusion gates ROM ≥ 60°, coverage ≥ 50 %, r ≥ 0.7);
6. saves `data/fit3d/feat_fit3d.npz` in the same format as `feat_rehab24.npz`.

**Licence:** Fit3D is non-commercial, single user, no redistribution. Keep the data and everything written here in a folder only you can read (`chmod -R 700`), never push it to GitHub.

In [1]:
# ---------------- settings: edit these ----------------
import os, sys, glob, json, time
# Windows PC example (raw strings r'...' because of the backslashes and spaces)
REPO = r'F:\namal sir project\github repo\AI-based-Physiotherapy-project-'            # <-- your repo folder
FIT3D_ROOT = r'F:\namal sir project\github repo\AI-based-Physiotherapy-project-\fit3d dataset\curls\train'        # folder holding s03, s04, ...
# ADA / Linux example:
# REPO = os.path.expanduser('~/AI-based-Physiotherapy-project-'); FIT3D_ROOT = os.path.expanduser('~/fit3d/train')
OUT        = os.path.join(REPO, 'data', 'fit3d')                       # git-ignored
EXERCISES  = ['dumbbell_biceps_curls', 'dumbbell_hammer_curls', 'drag_curl']
EXTRACT_ONLY = False     # True = stop after MediaPipe (e.g. on your laptop), then copy data/fit3d/cache
STEP       = 2          # every 2nd frame: 50 fps -> 25 fps (close to a 30 fps webcam)
N_WORKERS  = max(1, (os.cpu_count() or 2) - 1)
THRESHOLD, MAX_LAG, MIN_ROM, MIN_COV = 0.70, 30, 60.0, 50.0   # same as the REHAB24-6 benchmark

os.makedirs(os.path.join(OUT, 'cache'), exist_ok=True)
sys.path[:0] = [REPO, os.path.join(REPO, 'pose'), os.path.join(REPO, 'scripts')]
print('subjects:', sorted(os.path.basename(p) for p in glob.glob(os.path.join(FIT3D_ROOT, 's*'))))
print('workers :', N_WORKERS)

subjects: ['s03', 's04', 's05', 's07', 's08', 's09', 's10', 's11']
workers : 7


In [2]:
# ---------------- repo code (identical to the benchmark) ----------------
import inspect
import numpy as np, pandas as pd
import angles as angles_2d
from angles import elbow_angle_3d_ordered
from angles_lift import bone_lengths, elbow_angle_lifted
from angles_world import mask_world_like
from visibility import mask_low_visibility
from features import frame_features, overlap_rows, FEATURE_NAMES
import benchmark_common as bc

assert 'lfilter_zi' in inspect.getsource(bc.butterworth_filter), \
    'benchmark_common.butterworth_filter is not causal yet - run cell A of notebook 00 first'
CAUSAL = True

def fill_1d(arr, max_gap=5):
    '''Copied from the benchmark notebook (STEP 5) so both datasets are treated alike.'''
    arr = np.array(arr, dtype=float)
    if np.isnan(arr).all() or not np.isnan(arr).any():
        return arr
    idx = np.arange(len(arr))
    good = ~np.isnan(arr)
    filled = arr.copy()
    in_gap, gap_start = False, 0
    for i in range(len(arr)):
        if np.isnan(arr[i]) and not in_gap:
            in_gap, gap_start = True, i
        elif not np.isnan(arr[i]) and in_gap:
            in_gap = False
            if i - gap_start <= max_gap:
                filled[gap_start:i] = np.interp(idx[gap_start:i], idx[good], arr[good])
    return filled

def smooth(x, fps):
    x = fill_1d(x, 5)
    return bc.butterworth_filter(x, fs=fps, cutoff=6.0, causal=CAUSAL) if np.isfinite(x).sum() > 15 else x

## 1 — MediaPipe on every video (parallel, cached)

In [3]:
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mproc

def cache_path(subj, cam, ex):
    return os.path.join(OUT, 'cache', f'{subj}_{cam}_{ex}_s{STEP}_w.npz')

def extract_one(job):
    video, cache = job
    if os.path.exists(cache):
        return cache, 'cached', 0.0
    import cv2
    from landmarks import LandmarkExtractor
    t0 = time.time()
    cap = cv2.VideoCapture(video)
    fps = cap.get(cv2.CAP_PROP_FPS) or 50.0
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    ext = LandmarkExtractor(model_complexity=1)
    lms, wds, frames, i = [], [], [], 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % STEP == 0:
            n, w = ext.process_with_world(frame)
            lms.append(n); wds.append(w); frames.append(i)
        i += 1
    cap.release(); ext.close()
    np.savez_compressed(cache, lm=np.asarray(lms, np.float32), world=np.asarray(wds, np.float32),
                        frames=np.asarray(frames), size=np.asarray([W, H]), fps=fps / STEP)
    return cache, 'new', time.time() - t0

# Recordings are found from the camera files, so the videos are only needed for
# recordings that are not cached yet (you can run MediaPipe on your laptop and
# copy only data/fit3d/cache/ to the server).
jobs, index, missing = [], [], []
for sd in sorted(glob.glob(os.path.join(FIT3D_ROOT, 's*'))):
    subj = os.path.basename(sd)
    for ex in EXERCISES:
        for cp in sorted(glob.glob(os.path.join(sd, 'camera_parameters', '*', ex + '.json'))):
            cam = os.path.basename(os.path.dirname(cp))
            v, c = os.path.join(sd, 'videos', cam, ex + '.mp4'), cache_path(subj, cam, ex)
            if os.path.exists(c) or os.path.exists(v):
                jobs.append((v, c)); index.append((subj, cam, ex))
            else:
                missing.append(f'{subj}/{cam}/{ex}')
print(len(jobs), 'recordings;', sum(os.path.exists(c) for _, c in jobs), 'already cached;',
      len(missing), 'with neither video nor cache (skipped)')

todo = [j for j in jobs if not os.path.exists(j[1])]
t0 = time.time()
if todo and 'fork' in mproc.get_all_start_methods() and N_WORKERS > 1:      # Linux / ADA
    with ProcessPoolExecutor(N_WORKERS, mp_context=mproc.get_context('fork')) as pool:
        for k, (c, how, sec) in enumerate(pool.map(extract_one, todo), 1):
            print(f'[{k}/{len(todo)}] {os.path.basename(c)}  {sec:.0f}s', flush=True)
else:                                     # Windows: no fork in notebooks -> one by one
    for k, job in enumerate(todo, 1):
        c, how, sec = extract_one(job)
        print(f'[{k}/{len(todo)}] {os.path.basename(c)}  {sec:.0f}s', flush=True)
# stopped half way? just run this cell again - finished videos are cached and skipped
print(f'done in {(time.time() - t0) / 60:.1f} min')
if EXTRACT_ONLY:
    raise SystemExit('EXTRACT_ONLY: cache written to ' + os.path.join(OUT, 'cache') + ' - copy that folder to the server')

96 recordings; 0 already cached; 0 with neither video nor cache (skipped)


f:\namal sir project\github repo\AI-based-Physiotherapy-project-\.venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


[1/96] s03_50591643_dumbbell_biceps_curls_s2_w.npz  14s
[2/96] s03_58860488_dumbbell_biceps_curls_s2_w.npz  13s
[3/96] s03_60457274_dumbbell_biceps_curls_s2_w.npz  17s
[4/96] s03_65906101_dumbbell_biceps_curls_s2_w.npz  16s
[5/96] s03_50591643_dumbbell_hammer_curls_s2_w.npz  19s
[6/96] s03_58860488_dumbbell_hammer_curls_s2_w.npz  22s
[7/96] s03_60457274_dumbbell_hammer_curls_s2_w.npz  25s
[8/96] s03_65906101_dumbbell_hammer_curls_s2_w.npz  27s
[9/96] s03_50591643_drag_curl_s2_w.npz  35s
[10/96] s03_58860488_drag_curl_s2_w.npz  53s
[11/96] s03_60457274_drag_curl_s2_w.npz  53s
[12/96] s03_65906101_drag_curl_s2_w.npz  41s
[13/96] s04_50591643_dumbbell_biceps_curls_s2_w.npz  37s
[14/96] s04_58860488_dumbbell_biceps_curls_s2_w.npz  41s
[15/96] s04_60457274_dumbbell_biceps_curls_s2_w.npz  39s
[16/96] s04_65906101_dumbbell_biceps_curls_s2_w.npz  39s
[17/96] s04_50591643_dumbbell_hammer_curls_s2_w.npz  36s
[18/96] s04_58860488_dumbbell_hammer_curls_s2_w.npz  36s
[19/96] s04_60457274_dumbbell_h

## 2 — Which Fit3D joints are shoulder / elbow / wrist?

Instead of trusting a joint-order table, every recording projects all 25 mocap joints
into its own camera and picks, for each MediaPipe landmark, the joint that lands on it.
Four rotation conventions are tried; the one with the smallest pixel error wins.
The majority answer over all recordings is then used everywhere.

In [4]:
MP = {'left': (11, 13, 15), 'right': (12, 14, 16)}

def load_json(p):
    with open(p, encoding='utf-8') as f:
        return json.load(f)

def load_joints(p):
    d = load_json(p)
    a = np.asarray(d.get('joints3d_25', d) if isinstance(d, dict) else d, float)
    assert a.ndim == 3 and a.shape[1:] == (25, 3), a.shape
    return a

def load_camera(p):
    d = load_json(p)
    ex = d['extrinsics']
    it = d.get('intrinsics_wo_distortion') or d.get('intrinsics_w_distortion') or d.get('intrinsics')
    return (np.asarray(ex['R'], float).reshape(3, 3), np.asarray(ex['T'], float).reshape(3),
            np.asarray(it['f'], float).reshape(-1), np.asarray(it['c'], float).reshape(-1))

def to_camera(X, R, T, mode):
    return [(X - T) @ R.T, X @ R.T + T, (X - T) @ R, X @ R + T][mode]

def project(Xc, f, c):
    with np.errstate(invalid='ignore', divide='ignore'):
        return Xc[..., :2] / Xc[..., 2:3] * f + c

def match(J, lm_px, R, T, f, c):
    best = None
    for mode in range(4):
        Xc = to_camera(J, R, T, mode)
        if np.nanmedian(Xc[..., 2]) <= 0:
            continue
        P = project(Xc, f, c)
        mapping, errs = {}, []
        for side, idx in MP.items():
            pick = []
            for m in idx:
                med = np.nanmedian(np.linalg.norm(P - lm_px[:, m:m + 1, :2], axis=-1), axis=0)
                j = int(np.nanargmin(med)); pick.append(j); errs.append(med[j])
            mapping[side] = tuple(pick)
        e = float(np.nanmedian(errs))
        if best is None or e < best[2]:
            best = (mode, mapping, e)
    return best

def load_rec(subj, cam, ex):
    sd = os.path.join(FIT3D_ROOT, subj)
    c = np.load(cache_path(subj, cam, ex))
    joints = load_joints(os.path.join(sd, 'joints3d_25', ex + '.json'))
    frames = c['frames']; keep = frames < len(joints)
    return dict(lm=c['lm'][keep].astype(float), world=c['world'][keep].astype(float),
                frames=frames[keep], size=tuple(int(v) for v in c['size']), fps=float(c['fps']),
                J=joints[frames[keep]],
                cam=load_camera(os.path.join(sd, 'camera_parameters', cam, ex + '.json')))

print(len(index), 'recordings to check')
if not index:
    raise RuntimeError('Section 1 found no videos. Check FIT3D_ROOT (it must hold s03, s04, ...) '
                       'and that section 1 printed "N videos" with N > 0.')
votes, problems = [], []
for subj, cam, ex in index:
    try:
        r = load_rec(subj, cam, ex)
    except Exception as e:
        problems.append((subj, cam, ex, f'{type(e).__name__}: {e}')); continue
    seen = np.isfinite(r['lm'][:, [11, 12, 13, 14, 15, 16], 0]).all(axis=1)
    if seen.sum() < 20:
        problems.append((subj, cam, ex, f'MediaPipe found the arms in only {int(seen.sum())} frames')); continue
    px = r['lm'][seen].copy(); px[..., 0] *= r['size'][0]; px[..., 1] *= r['size'][1]
    res = match(r['J'][seen][::5], px[::5], *r['cam'])
    if res is None:
        problems.append((subj, cam, ex, 'joints project behind the camera in every convention')); continue
    mode, mapping, err = res
    votes.append((subj, cam, ex, mode, str(mapping['left']), str(mapping['right']), round(err, 1)))

if problems:
    print('skipped while matching:')
    for p in problems[:20]:
        print('  ', *p)
V = pd.DataFrame(votes, columns=['subject', 'camera', 'exercise', 'mode', 'left', 'right', 'px_err'])
if V.empty:
    raise RuntimeError('No recording could be matched - send the "skipped while matching" lines above.')
key = V.groupby(['mode', 'left', 'right']).size().sort_values(ascending=False)
print(key.head())
(MODE, LEFT, RIGHT), n_agree = key.index[0], int(key.iloc[0])
MODE, LEFT, RIGHT = int(MODE), tuple(int(v) for v in LEFT.strip('()').split(',')), \
                    tuple(int(v) for v in RIGHT.strip('()').split(','))
JMAP = {'left': LEFT, 'right': RIGHT}
good = V[(V['mode'] == MODE) & (V['left'] == str(LEFT)) & (V['right'] == str(RIGHT))]
print(f'\nusing mode {MODE}, left {LEFT}, right {RIGHT}  ({n_agree}/{len(V)} recordings agree; '
      f'median px error {good.px_err.median():.1f})')
if n_agree < 0.6 * len(V) or good.px_err.median() > 40:
    print('WARNING: weak agreement or large pixel error - send this table before trusting the results')

96 recordings to check
mode  left          right       
0     (11, 12, 13)  (14, 15, 16)    94
      (11, 0, 13)   (14, 15, 16)     1
      (11, 12, 13)  (14, 15, 23)     1
dtype: int64

using mode 0, left (11, 12, 13), right (14, 15, 16)  (94/96 recordings agree; median px error 8.1)


## 3 — Truth, the three angle variants and the 20 features, per arm

In [5]:
def view_of(J, R, T):
    Xc = to_camera(J, R, T, MODE)
    v = Xc[:, JMAP['left'][0]] - Xc[:, JMAP['right'][0]]
    yaw = float(np.nanmedian(np.degrees(np.arctan2(np.abs(v[:, 2]), np.linalg.norm(v[:, :2], axis=1)))))
    return yaw, 'front' if yaw < 30 else 'half' if yaw < 60 else 'profile'

rows, parts, skipped = [], [], {'static': 0, 'occluded': 0, 'align': 0}
for subj, cam, ex in index:
    r = load_rec(subj, cam, ex)
    lm, world, size, fps = r['lm'], r['world'], r['size'], r['fps']
    R, T, f, c = r['cam']
    yaw, view = view_of(r['J'], R, T)
    lm_m = mask_low_visibility(lm, threshold=THRESHOLD)
    wd_m = mask_world_like(world, lm_m)
    for side in ('right', 'left'):
        s, e, w = JMAP[side]
        gt = angles_2d.angle(r['J'][:, s], r['J'][:, e], r['J'][:, w])
        L_up, L_fo = bone_lengths(lm_m, side, size)
        pa = smooth(angles_2d.elbow_angle(lm_m, side=side, size=size), fps)
        pb = smooth(elbow_angle_3d_ordered(wd_m, side=side), fps)
        pe = smooth(elbow_angle_lifted(lm_m, wd_m, side, size, L_up, L_fo), fps)
        feat = frame_features(lm_m, wd_m, side, size, L_up, L_fo, pa, pb, pe)

        rom = float(np.nanpercentile(gt, 95) - np.nanpercentile(gt, 5))
        cov = 100.0 * float(np.isfinite(pa).mean())
        if rom < MIN_ROM:
            skipped['static'] += 1; continue
        if cov < MIN_COV:
            skipped['occluded'] += 1; continue
        lag, lag_r = bc.best_lag(pa, gt, max_lag=MAX_LAG, filter_signals=True, fs=fps)
        if abs(lag) >= MAX_LAG or lag_r < 0.7:
            skipped['align'] += 1; continue

        F, g = overlap_rows(feat, gt, lag)
        ok = np.isfinite(F[:, 0]) & np.isfinite(g)
        rec = f'fit3d_{subj}_{cam}_{ex}_{side}'
        parts.append((F[ok], g[ok], subj, rec, view, ex))
        rows.append(dict(recording=rec, subject=subj, camera=cam, exercise=ex, side=side,
                         view=view, yaw=round(yaw, 1), lag=lag, r=round(lag_r, 3), frames=int(ok.sum()),
                         raw_mae=round(float(np.mean(np.abs(F[ok, 0] - g[ok]))), 2),
                         truth_p5=round(float(np.percentile(g[ok], 5)), 1),
                         truth_p95=round(float(np.percentile(g[ok], 95)), 1)))
REC = pd.DataFrame(rows)
print('kept', len(REC), 'arm-recordings; skipped', skipped)
REC.groupby('view').agg(recordings=('recording', 'size'), frames=('frames', 'sum'),
                        raw_mae_median=('raw_mae', 'median'),
                        truth_p5=('truth_p5', 'median'), truth_p95=('truth_p95', 'median'))

F:\namal sir project\github repo\AI-based-Physiotherapy-project-\angles_lift.py:238: RuntimeWarning: All-NaN slice encountered
  out = np.nanmedian(stack, axis=0)
f:\namal sir project\github repo\AI-based-Physiotherapy-project-\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


kept 129 arm-recordings; skipped {'static': 0, 'occluded': 58, 'align': 5}


,recordings,frames,raw_mae_median,truth_p5,truth_p95
view,,,,,
front,123,54944,20.33,49.30,169.0
half,6,3401,20.36,49.35,169.7


In [7]:
out = os.path.join(OUT, 'feat_fit3d.npz')
n = [len(p[1]) for p in parts]
np.savez_compressed(
    out,
    X=np.concatenate([p[0] for p in parts]).astype(np.float32),
    y=np.concatenate([p[1] for p in parts]).astype(np.float32),
    feature_names=np.array(FEATURE_NAMES),
    person=np.repeat([f'fit3d_{p[2]}' for p in parts], n),
    recording=np.repeat([p[3] for p in parts], n),
    view=np.repeat([p[4] for p in parts], n),
    exercise=np.repeat([p[5] for p in parts], n),
    dataset=np.array(['fit3d'] * sum(n)),
    causal=np.array(True),
)
REC.to_csv(os.path.join(OUT, 'fit3d_recordings.csv'), index=False)
print(f'saved {out}: {sum(n):,} frames from {len(parts)} arm-recordings')

saved F:\namal sir project\github repo\AI-based-Physiotherapy-project-\data\fit3d\feat_fit3d.npz: 58,345 frames from 129 arm-recordings
